# TAAT, DAAT, and BM25 over Postings

Boolean retrieval returns sets; ranked retrieval returns scored lists. This demo
implements two evaluation strategies, Term-at-a-Time (TAAT) and
Document-at-a-Time (DAAT), over a real inverted index built from the 50-book
library collection. Both produce identical BM25 rankings, but differ in how they
traverse postings and manage memory.

**Learning goals:**
- Implement Term-at-a-Time (TAAT) and Document-at-a-Time (DAAT) evaluation
- Score documents with BM25 using only postings-list data
- Decompose BM25 scores into per-term contributions
- Understand trade-offs: memory footprint vs. pruning opportunities
- Explore how BM25 parameters k1 and b change rankings

**Prerequisites:** Chapter 1 section 1.5 (BM25 formula), ch04-01 (inverted index structure)

In [1]:
import math
from collections import Counter
from shared.library_collection import LIBRARY, texts, doc2text
from shared.text import pipeline
from shared.retrieval import document_frequencies, average_doc_length, idf_bm25
from shared.display import print_table, display_md

## Build the Index

We index the library collection using title and author as searchable text.
The pipeline applies lowercasing, stopword removal, and stemming.

In [2]:
# Build corpus: {doc_id: [stemmed tokens]}
corpus = {}
for key, record in LIBRARY.items():
    text = f"{record['title']} {record['author']}"
    corpus[key] = pipeline(text)

N = len(corpus)
df = document_frequencies(corpus)
adl = average_doc_length(corpus)

display_md(
    f"| Statistic | Value |\n"
    f"|-----------|-------|\n"
    f"| Documents | {N} |\n"
    f"| Vocabulary | {len(df)} terms |\n"
    f"| Avg. doc length | {adl:.2f} tokens |"
)

| Statistic | Value |
|-----------|-------|
| Documents | 50 |
| Vocabulary | 231 terms |
| Avg. doc length | 5.34 tokens |

In [3]:
# Build inverted index: {term: [(doc_id, tf), ...]}
inverted_index = {}
doc_lengths = {}

for doc_id, tokens in corpus.items():
    doc_lengths[doc_id] = len(tokens)
    tf_counts = Counter(tokens)
    for term, tf in tf_counts.items():
        if term not in inverted_index:
            inverted_index[term] = []
        inverted_index[term].append((doc_id, tf))

# Sort postings by doc_id
for term in inverted_index:
    inverted_index[term].sort(key=lambda x: x[0])

display_md(f"**Index built:** {len(inverted_index)} terms, postings sorted by doc_id.")

**Index built:** 231 terms, postings sorted by doc_id.

## Define a Query

We will search for books about "computer systems design", a query that spans
multiple terms and should surface several relevant titles.

In [4]:
query = "computer systems design"
q_tokens = pipeline(query)

rows = []
for t in q_tokens:
    postings = inverted_index.get(t, [])
    idf_val = idf_bm25(t, df, N)
    rows.append([t, df.get(t, 0), f"{idf_val:.4f}", len(postings)])

display_md(f"**Query:** `{query}` -> tokens: `{q_tokens}`\n")
print_table(rows, headers=["Term", "DF", "IDF (BM25)", "Postings length"])

**Query:** `computer systems design` -> tokens: `['comput', 'system', 'design']`


| Term   |   DF |   IDF (BM25) |   Postings length |
|:-------|-----:|-------------:|------------------:|
| comput |    5 |       2.113  |                 5 |
| system |    3 |       2.608  |                 3 |
| design |    2 |       2.9653 |                 2 |

## Term-at-a-Time (TAAT)

TAAT processes one query term at a time. For each term, we walk its entire
postings list and add partial BM25 scores to an accumulator array.

In [5]:
def bm25_taat(q_tokens, inverted_index, doc_lengths, df, N, adl, k1=1.2, b=0.75):
    """
    Term-at-a-Time BM25 scoring.

    For each query term, iterate its full postings list and accumulate
    partial scores in a dictionary keyed by doc_id.
    """
    accumulators = {}  # {doc_id: cumulative_score}
    postings_accessed = 0

    for term in set(q_tokens):
        if term not in inverted_index:
            continue
        idf_val = idf_bm25(term, df, N)
        postings = inverted_index[term]

        for doc_id, tf in postings:
            postings_accessed += 1
            dl = doc_lengths[doc_id]
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * dl / adl)
            score_contrib = idf_val * numerator / denominator

            accumulators[doc_id] = accumulators.get(doc_id, 0.0) + score_contrib

    # Sort by score descending
    ranked = sorted(accumulators.items(), key=lambda x: -x[1])
    return ranked, postings_accessed, len(accumulators)

display_md(
    "**TAAT approach:**\n\n"
    "1. For each query term, read its FULL postings list\n"
    "2. For each posting (doc_id, tf), compute partial BM25 and add to accumulator\n"
    "3. After all terms processed, sort accumulators by score"
)

**TAAT approach:**

1. For each query term, read its FULL postings list
2. For each posting (doc_id, tf), compute partial BM25 and add to accumulator
3. After all terms processed, sort accumulators by score

In [6]:
# Run TAAT
taat_results, taat_postings, taat_accumulators = bm25_taat(
    q_tokens, inverted_index, doc_lengths, df, N, adl
)

display_md(
    f"**TAAT results for** `{query}`\n\n"
    f"- Postings accessed: {taat_postings}\n"
    f"- Accumulators created: {taat_accumulators}\n"
    f"- Top 10 results:"
)

rows = []
for rank, (doc_id, score) in enumerate(taat_results[:10], 1):
    title = LIBRARY[doc_id]["title"]
    rows.append([rank, doc_id, f"{score:.4f}", title[:50]])

print_table(rows, headers=["Rank", "Doc ID", "BM25 Score", "Title"])

**TAAT results for** `computer systems design`

- Postings accessed: 10
- Accumulators created: 8
- Top 10 results:

|   Rank | Doc ID                |   BM25 Score | Title                                              |
|-------:|:----------------------|-------------:|:---------------------------------------------------|
|      1 | computer-organization |       3.5423 | Computer Organization and Design: The Hardware/Sof |
|      2 | computer-systems      |       3.479  | Computer Systems: A Programmer's Perspective       |
|      3 | everyday-things       |       3.3045 | The Design of Everyday Things                      |
|      4 | theory-of-computation |       2.1695 | Introduction to the Theory of Computation          |
|      5 | computer-networks     |       2.0113 | Computer Networks                                  |
|      6 | operating-systems     |       1.9219 | Operating System Concepts                          |
|      7 | database-systems      |       1.8192 | Database Systems: The Complete Book                |
|      8 | sicp                  |       1.6503 | Structure and Interpretation of Computer Programs  |

## Document-at-a-Time (DAAT)

DAAT processes one document at a time. We advance pointers across all query-term
postings lists simultaneously, always processing the smallest doc_id next.

In [7]:
def bm25_daat(q_tokens, inverted_index, doc_lengths, df, N, adl, k1=1.2, b=0.75):
    """
    Document-at-a-Time BM25 scoring.

    Maintain a pointer into each query term's postings list.
    At each step, find the smallest doc_id across all pointers,
    collect all term contributions for that doc, then advance.
    """
    # Initialize pointers for each query term
    terms = [t for t in set(q_tokens) if t in inverted_index]
    postings_lists = {t: inverted_index[t] for t in terms}
    pointers = {t: 0 for t in terms}
    idf_vals = {t: idf_bm25(t, df, N) for t in terms}

    results = []
    postings_accessed = 0

    while True:
        # Find the smallest doc_id among all active pointers
        min_doc = None
        for t in terms:
            if pointers[t] < len(postings_lists[t]):
                doc_id = postings_lists[t][pointers[t]][0]
                if min_doc is None or doc_id < min_doc:
                    min_doc = doc_id

        if min_doc is None:
            break  # All lists exhausted

        # Collect scores from all terms present in this document
        score = 0.0
        for t in terms:
            if pointers[t] < len(postings_lists[t]):
                doc_id, tf = postings_lists[t][pointers[t]]
                if doc_id == min_doc:
                    postings_accessed += 1
                    dl = doc_lengths[doc_id]
                    numerator = tf * (k1 + 1)
                    denominator = tf + k1 * (1 - b + b * dl / adl)
                    score += idf_vals[t] * numerator / denominator
                    pointers[t] += 1

        results.append((min_doc, score))

    # Sort by score descending
    results.sort(key=lambda x: -x[1])
    return results, postings_accessed

display_md(
    "**DAAT approach:**\n\n"
    "1. Maintain a pointer into each term's postings list\n"
    "2. Find the smallest doc_id across all pointers\n"
    "3. Collect BM25 contributions from all terms present in that doc\n"
    "4. Advance pointers past that doc_id\n"
    "5. Repeat until all lists exhausted"
)

**DAAT approach:**

1. Maintain a pointer into each term's postings list
2. Find the smallest doc_id across all pointers
3. Collect BM25 contributions from all terms present in that doc
4. Advance pointers past that doc_id
5. Repeat until all lists exhausted

In [8]:
# Run DAAT
daat_results, daat_postings = bm25_daat(
    q_tokens, inverted_index, doc_lengths, df, N, adl
)

display_md(f"**DAAT results for** `{query}`\n\n- Postings accessed: {daat_postings}\n- Top 10:")

rows = []
for rank, (doc_id, score) in enumerate(daat_results[:10], 1):
    title = LIBRARY[doc_id]["title"]
    rows.append([rank, doc_id, f"{score:.4f}", title[:50]])

print_table(rows, headers=["Rank", "Doc ID", "BM25 Score", "Title"])

**DAAT results for** `computer systems design`

- Postings accessed: 10
- Top 10:

|   Rank | Doc ID                |   BM25 Score | Title                                              |
|-------:|:----------------------|-------------:|:---------------------------------------------------|
|      1 | computer-organization |       3.5423 | Computer Organization and Design: The Hardware/Sof |
|      2 | computer-systems      |       3.479  | Computer Systems: A Programmer's Perspective       |
|      3 | everyday-things       |       3.3045 | The Design of Everyday Things                      |
|      4 | theory-of-computation |       2.1695 | Introduction to the Theory of Computation          |
|      5 | computer-networks     |       2.0113 | Computer Networks                                  |
|      6 | operating-systems     |       1.9219 | Operating System Concepts                          |
|      7 | database-systems      |       1.8192 | Database Systems: The Complete Book                |
|      8 | sicp                  |       1.6503 | Structure and Interpretation of Computer Programs  |

## TAAT vs. DAAT: Same Rankings, Different Process

Both strategies produce identical scores and rankings. The difference is in
how they organize memory and traversal.

In [9]:
# Verify identical top-10 rankings
taat_top10 = [(d, round(s, 6)) for d, s in taat_results[:10]]
daat_top10 = [(d, round(s, 6)) for d, s in daat_results[:10]]

match = all(t[0] == d[0] and t[1] == d[1] for t, d in zip(taat_top10, daat_top10))

display_md(
    f"**Verification:** Rankings identical? **{match}**\n\n"
    f"| Property | TAAT | DAAT |\n"
    f"|----------|------|------|\n"
    f"| Processing unit | One term at a time | One document at a time |\n"
    f"| Postings accessed | {taat_postings} | {daat_postings} |\n"
    f"| Memory (accumulators) | {taat_accumulators} (all matching docs) | 1 (current doc only) |\n"
    f"| Early termination | Difficult | Possible (skip low-scoring docs) |\n"
    f"| Best for | Short queries | Long queries with pruning |"
)

**Verification:** Rankings identical? **True**

| Property | TAAT | DAAT |
|----------|------|------|
| Processing unit | One term at a time | One document at a time |
| Postings accessed | 10 | 10 |
| Memory (accumulators) | 8 (all matching docs) | 1 (current doc only) |
| Early termination | Difficult | Possible (skip low-scoring docs) |
| Best for | Short queries | Long queries with pruning |

## BM25 Score Decomposition

Let us look inside the top-ranked document and see exactly how BM25 assembles
its score from per-term contributions.

In [10]:
# Decompose the top document's score
top_doc_id = taat_results[0][0]
top_record = LIBRARY[top_doc_id]
top_tokens = corpus[top_doc_id]
top_tf = Counter(top_tokens)
top_dl = doc_lengths[top_doc_id]

k1, b = 1.2, 0.75

rows = []
total = 0.0
for term in set(q_tokens):
    tf_val = top_tf.get(term, 0)
    idf_val = idf_bm25(term, df, N)
    if tf_val > 0:
        numerator = tf_val * (k1 + 1)
        denominator = tf_val + k1 * (1 - b + b * top_dl / adl)
        contrib = idf_val * numerator / denominator
    else:
        numerator = denominator = contrib = 0.0
    total += contrib
    rows.append([term, tf_val, f"{idf_val:.4f}", f"{numerator:.4f}",
                 f"{denominator:.4f}", f"{contrib:.4f}"])

display_md(
    f"**Score breakdown for:** {top_record['title']}\n\n"
    f"- Doc length: {top_dl} tokens (avg: {adl:.2f})\n"
    f"- Parameters: k1={k1}, b={b}\n"
)
print_table(rows, headers=["Term", "TF", "IDF", "Numerator", "Denominator", "Contribution"])
display_md(f"\n**Total BM25 score:** {total:.4f}")

**Score breakdown for:** Computer Organization and Design: The Hardware/Software Interface

- Doc length: 11 tokens (avg: 5.34)
- Parameters: k1=1.2, b=0.75


| Term   |   TF |    IDF |   Numerator |   Denominator |   Contribution |
|:-------|-----:|-------:|------------:|--------------:|---------------:|
| design |    1 | 2.9653 |         2.2 |        3.1539 |         2.0684 |
| system |    0 | 2.608  |         0   |        0      |         0      |
| comput |    1 | 2.113  |         2.2 |        3.1539 |         1.4739 |


**Total BM25 score:** 3.5423

## Parameter Exploration: k1 and b

BM25 has two key tuning parameters:
- **k1** controls TF saturation (higher = more credit for repeated terms)
- **b** controls length normalization (0 = no normalization, 1 = full normalization)

In [11]:
# Vary k1 with b fixed at 0.75
display_md("### Effect of k1 (b = 0.75 fixed)\n\nHigher k1 gives more weight to term repetition.")

k1_values = [0.5, 1.2, 2.0, 5.0]
rows = []

for k1_val in k1_values:
    results, _, _ = bm25_taat(q_tokens, inverted_index, doc_lengths, df, N, adl, k1=k1_val, b=0.75)
    top5_ids = [doc_id for doc_id, _ in results[:5]]
    top5_str = ", ".join(top5_ids[:3])
    rows.append([f"k1={k1_val}", f"{results[0][1]:.4f}", top5_str])

print_table(rows, headers=["Parameters", "Top Score", "Top-3 Doc IDs"])

### Effect of k1 (b = 0.75 fixed)

Higher k1 gives more weight to term repetition.

| Parameters   |   Top Score | Top-3 Doc IDs                                            |
|:-------------|------------:|:---------------------------------------------------------|
| k1=0.5       |      4.0145 | computer-organization, computer-systems, everyday-things |
| k1=1.2       |      3.5423 | computer-organization, computer-systems, everyday-things |
| k1=2.0       |      3.3907 | everyday-things, computer-organization, computer-systems |
| k1=5.0       |      3.5168 | everyday-things, computer-systems, computer-organization |

In [12]:
# Vary b with k1 fixed at 1.2
display_md("### Effect of b (k1 = 1.2 fixed)\n\nb=0 ignores document length; b=1 fully normalizes by length.")

b_values = [0.0, 0.25, 0.5, 0.75, 1.0]
rows = []

for b_val in b_values:
    results, _, _ = bm25_taat(q_tokens, inverted_index, doc_lengths, df, N, adl, k1=1.2, b=b_val)
    # Show top-5 with scores
    top5 = [(doc_id, score) for doc_id, score in results[:5]]
    top5_str = " | ".join(f"{d} ({s:.3f})" for d, s in top5[:3])
    rows.append([f"b={b_val}", top5_str])

print_table(rows, headers=["Parameters", "Top-3 (doc_id, score)"])

### Effect of b (k1 = 1.2 fixed)

b=0 ignores document length; b=1 fully normalizes by length.

| Parameters   | Top-3 (doc_id, score)                                                              |
|:-------------|:-----------------------------------------------------------------------------------|
| b=0.0        | computer-organization (5.078) | computer-systems (4.721) | everyday-things (2.965) |
| b=0.25       | computer-organization (4.437) | computer-systems (4.219) | everyday-things (3.070) |
| b=0.5        | computer-organization (3.939) | computer-systems (3.813) | everyday-things (3.183) |
| b=0.75       | computer-organization (3.542) | computer-systems (3.479) | everyday-things (3.304) |
| b=1.0        | everyday-things (3.436) | computer-organization (3.218) | computer-systems (3.198) |

In [13]:
# Show how rank positions shift with different b values
display_md("### Rank Changes Across b Values\n\nDocuments with different lengths react differently to length normalization.")

# Get docs that appear in top-10 for any b setting
all_top_docs = set()
b_rankings = {}
for b_val in b_values:
    results, _, _ = bm25_taat(q_tokens, inverted_index, doc_lengths, df, N, adl, k1=1.2, b=b_val)
    b_rankings[b_val] = {doc_id: rank for rank, (doc_id, _) in enumerate(results, 1)}
    all_top_docs.update(doc_id for doc_id, _ in results[:8])

rows = []
for doc_id in sorted(all_top_docs, key=lambda d: b_rankings[0.75].get(d, 99)):
    dl = doc_lengths[doc_id]
    rank_row = [doc_id, dl]
    for b_val in b_values:
        rank = b_rankings[b_val].get(doc_id, "-")
        rank_row.append(rank)
    rows.append(rank_row)

headers = ["Doc ID", "Length"] + [f"b={b}" for b in b_values]
print_table(rows[:10], headers=headers)

### Rank Changes Across b Values

Documents with different lengths react differently to length normalization.

| Doc ID                |   Length |   b=0.0 |   b=0.25 |   b=0.5 |   b=0.75 |   b=1.0 |
|:----------------------|---------:|--------:|---------:|--------:|---------:|--------:|
| computer-organization |       11 |       1 |        1 |       1 |        1 |       2 |
| computer-systems      |       10 |       2 |        2 |       2 |        2 |       3 |
| everyday-things       |        4 |       3 |        3 |       3 |        3 |       1 |
| theory-of-computation |        5 |       8 |        6 |       4 |        4 |       4 |
| computer-networks     |        6 |       6 |        7 |       6 |        5 |       5 |
| operating-systems     |       10 |       5 |        4 |       5 |        6 |       6 |
| database-systems      |       11 |       4 |        5 |       7 |        7 |       7 |
| sicp                  |        9 |       7 |        8 |       8 |        8 |       8 |

---

## Summary

| Concept | Key Point |
|---------|----------|
| TAAT | Process one term's full postings, accumulate across docs |
| DAAT | Process one doc at a time, advance all pointers together |
| Same result | Both strategies produce identical BM25 rankings |
| Memory trade-off | TAAT needs accumulators for all matching docs |
| Pruning trade-off | DAAT can skip docs that cannot reach top-k |
| k1 parameter | Controls TF saturation (typical: 1.2) |
| b parameter | Controls length normalization (typical: 0.75) |
| BM25 additivity | Score = sum of per-term contributions (enables both strategies) |

---

## Try It Yourself

1. Try a longer query (4-5 terms). Does DAAT access fewer postings than TAAT?
   When would the difference be significant?

2. Find a query where changing b from 0.0 to 1.0 flips the top-ranked document.
   Hint: look for cases where a short document and a long document both match.

3. Implement MaxScore pruning for DAAT: pre-compute the maximum possible
   contribution of each term, and skip documents that cannot reach the current
   k-th best score.

In [14]:
# Exercise: Try a different query and compare TAAT vs DAAT
exercise_query = "artificial intelligence modern approach"
exercise_tokens = pipeline(exercise_query)

taat_res, taat_p, taat_a = bm25_taat(exercise_tokens, inverted_index, doc_lengths, df, N, adl)
daat_res, daat_p = bm25_daat(exercise_tokens, inverted_index, doc_lengths, df, N, adl)

display_md(
    f"**Query:** `{exercise_query}` -> {exercise_tokens}\n\n"
    f"- TAAT postings accessed: {taat_p}, accumulators: {taat_a}\n"
    f"- DAAT postings accessed: {daat_p}\n"
    f"- Top result: {LIBRARY[taat_res[0][0]]['title']} (score: {taat_res[0][1]:.4f})"
)

**Query:** `artificial intelligence modern approach` -> ['artifici', 'intellig', 'modern', 'approach']

- TAAT postings accessed: 4, accumulators: 1
- DAAT postings accessed: 4
- Top result: Artificial Intelligence: A Modern Approach (score: 11.6184)